In [2]:
import pandas as pd
from h3 import h3
import glob
import numpy as np
from pandas import merge
import json
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import shape
from datetime import datetime, timedelta
import seaborn as sns

# Part 1: Derive Relocation Rules from Historical Data

In [11]:
# Pandas Config
pd.set_option('display.max_columns', None)
pd.options.display.max_rows = 200

# ShareNow Data (results from Prepare_Trip_Data)
VEHICLE_TRIP = '/Users/demircanm/sciebo/IS3/Research_Projects/HeterogeneousFleetManagement/01_Fleet_Sizing/0_Data/Berlin/shared_vehicle_trips_berlin.csv'

# TIMEZONE
TIMEZONE = 'Europe/Berlin'


In [12]:
# Read Trips
trips = pd.read_csv(VEHICLE_TRIP)

/Users/demircanm/opt/anaconda3/envs/SharedMobility/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3146: DtypeWarning: Columns (4,21,22) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [66]:
# Clean Vehicle Type
trips["vehicleType"] = trips["vehicleType"].replace('bycicle', 'bicycle')

In [68]:
hex_start_column, hex_end_column = "h3_start_7", "h3_end_7"
day_of_week_column = "is_holiday"
date_column = "date_start"
vehicle_type = "vehicleType"



In [69]:
# Count outflow trips in spatio temporal resolution
outflow = trips.groupby([date_column, vehicle_type, day_of_week_column, hex_start_column]).agg({"id": "count"}).reset_index()

outflow.rename(columns={"id":"count"}, inplace=True)

# Reduce to WorkingDay/Non-Working-Day
outflow = outflow.groupby([day_of_week_column, vehicle_type, hex_start_column]).agg({"count":"median"})

In [70]:
# Count outflow trips in spatio temporal resolution
inflow = trips.groupby([date_column, vehicle_type, day_of_week_column, hex_end_column]).agg({"id": "count"}).reset_index()

inflow.rename(columns={"id":"count"}, inplace=True)

# Reduce to WorkingDay/Non-Working-Day
inflow = inflow.groupby([day_of_week_column, vehicle_type, hex_end_column]).agg({"count":"median"})

In [73]:
# Consider only hexagons that have data for each temporal data bucket
hexagon_to_consider = trips.groupby([hex_start_column]).agg({"id": "count"}).reset_index()
hexagon_to_consider["key"] = 1 # For Cross Join Later

temporal_dimension_to_consider = trips.groupby([day_of_week_column]).agg({"id": "count"}).reset_index()
temporal_dimension_to_consider["key"] = 1 # For Cross Join Later

vehicle_type_to_consider = trips.groupby([vehicle_type]).agg({"id": "count"}).reset_index()
vehicle_type_to_consider["key"] = 1 # For Cross Join Later

result = hexagon_to_consider.merge(temporal_dimension_to_consider, on="key", how="inner")
result = result.merge(vehicle_type_to_consider, on="key", how="inner")

result = result.drop(columns={"id_x", "key", "id_y", "id"})

In [74]:
result

,h3_start_7,is_holiday,vehicleType
0,871f18840ffffff,False,bicycle
1,871f18840ffffff,False,car
2,871f18840ffffff,False,kick scooter
3,871f18840ffffff,False,scooter
4,871f18840ffffff,True,bicycle
...,...,...,...
1059,871f1d591ffffff,False,scooter
1060,871f1d591ffffff,True,bicycle
1061,871f1d591ffffff,True,car
1062,871f1d591ffffff,True,kick scooter


In [75]:
# Merge Inflow
result = result.merge(inflow, left_on=[hex_start_column, vehicle_type, day_of_week_column], right_on=[hex_end_column, vehicle_type, day_of_week_column], how = "left")

result = result.rename(columns={"count":"inflow"})

# Merge Outflow
result = result.merge(outflow, left_on=[hex_start_column, vehicle_type, day_of_week_column], right_on=[hex_start_column, vehicle_type, day_of_week_column], how = "left")

result = result.rename(columns={"count":"outflow"})

# Fill NA
result = result.fillna(0)

In [76]:
result["diff"] = result["inflow"] - result["outflow"]

In [81]:
# Not used in Simulation so far...
result.to_csv('inflow_outflow_hex_7_daily.csv', index = False)

# Part 2: Derive Relocation Rules from Simulation Data

In [11]:
# Pandas Config
pd.set_option('display.max_columns', None)
pd.options.display.max_rows = 200

# ShareNow Data (results from Prepare_Trip_Data)
TRANSACTIONS = '../../data/output/transactions.csv'

# TIMEZONE
TIMEZONE = 'Europe/Berlin'

In [12]:
# Read Trips
trips = pd.read_csv(TRANSACTIONS)

In [13]:
trips

,origin,destination,time_start,time_end,price,distance,duration,selected_vehicle_mode,state,SharedCar,SharedKickScooter,SharedScooter
0,871f1d4d2ffffff,871f1d4d0ffffff,1.594855,1.594855,0.000000,1173.466260,0.000000,None,Missed,0,1,0
1,871f18b24ffffff,871f18b24ffffff,0.151785,3.987756,0.127048,222.888665,0.668673,SharedKickScooter,Fulfilled,0,1,0
2,871f18b20ffffff,871f18b24ffffff,4.768195,4.768195,0.000000,2507.841293,0.000000,None,Missed,1,0,1
3,871f1d4d4ffffff,871f1d4d4ffffff,0.536715,5.867576,0.625913,1098.081536,3.294278,SharedKickScooter,Fulfilled,0,1,0
4,871f1d48bffffff,871f1d48bffffff,0.337884,5.936408,0.840016,1826.122138,3.652244,SharedScooter,Fulfilled,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
15461,871f1d4d3ffffff,871f18b21ffffff,1431.970644,1439.641240,1.353442,4337.936335,5.205544,SharedCar,Fulfilled,1,0,1
15462,871f1d4d2ffffff,871f18b21ffffff,1439.687134,1439.687134,0.000000,1110.113453,0.000000,None,Missed,0,1,0
15463,871f1d483ffffff,871f1d48bffffff,1426.907392,1439.829643,1.682977,5394.136130,6.472989,SharedCar,Fulfilled,1,0,0
15464,871f18b25ffffff,871f18b24ffffff,1429.904112,1439.908304,1.311197,4202.536751,5.043064,SharedCar,Fulfilled,1,0,1


In [221]:
ordered_available_vehicle = ["SharedKickScooter", "SharedScooter", "SharedCar"]
# Rearrange Columns according to Vehicle Preferences
cols = trips.columns.tolist()
cols = cols[:-(len(ordered_available_vehicle))] + ordered_available_vehicle
cols

# Rearrange Columns
trips = trips[cols]

trips["cheapest_vehicle_mode"] = ''

for i in range(len(ordered_available_vehicle)):
    query = '{} == {}'.format(ordered_available_vehicle[i], 1)
    for k in range(i):
        query += ' & {} == {}'.format(ordered_available_vehicle[k], 0)
    
    trips.loc[trips.query(query).index, "cheapest_vehicle_mode"] = ordered_available_vehicle[i]

In [134]:
# (All) Hexagons
hexagons = pd.unique(trips[["origin", "destination"]].values.ravel('K'))
hexagons = pd.DataFrame(hexagons, columns=["hex_id"])

In [272]:
# Calculate Expected Outflow 
outflow = trips.groupby(["origin", "state", "cheapest_vehicle_mode"]).agg({"destination":"count"})

# Unstack Vehicle Type Level 
outflow = outflow.unstack()
vehicle_columns = list(outflow["destination"].columns.values)
# Reset Multi-Index to Flat Index
outflow = outflow.reset_index()
outflow.columns = ["origin", "state"] + vehicle_columns

# to ensure that "fulfilled" comes before "missed" during unstacking
outflow = outflow.sort_values(["origin", "state"])

# Unstack State Column
outflow = outflow.set_index(["origin", "state"])
outflow = outflow.unstack().reset_index()

# Arrange Column Names
unfolded_cols = ["hex_id"] 
for vehicle in vehicle_columns:
    unfolded_cols = unfolded_cols + ["OutFlow_Fulfilled_{}".format(vehicle), "OutFlow_Missed_{}".format(vehicle)]

# Overwrite Column Names
outflow.columns = unfolded_cols

In [282]:
# Calculate Expected Inflow
inflow = trips.groupby(["destination", "state", "cheapest_vehicle_mode"]).agg({"origin":"count"})

# Unstack Vehicle Type Level
inflow = inflow.unstack()
base_column = ["destination", "state"]
vehicle_columns = list(inflow["origin"].columns.values)

# Reset Multi-Index to Flat Index (keep vehicle column order)
inflow = inflow.reset_index()
inflow.columns = ["hex_id", "state"] + ["Inflow_{}".format(vehicle) for vehicle in vehicle_columns]

# to ensure that "fulfilled" comes before "missed" during unstacking
inflow = inflow.sort_values(["hex_id", "state"])

# Inflow, we just need fulfilled trips to estimate expected inflow
inflow = inflow[inflow["state"]=="Fulfilled"]
inflow = inflow.drop(columns={"state"})
inflow = inflow.reset_index()

In [284]:
# Put together expected outflow and inflow
result = hexagons.merge(inflow, on = ["hex_id"], how = "left", suffixes = ("",""))
result = result.merge(outflow, on = ["hex_id"], how = "left", suffixes = ("_inflow", "_outflow"))
result.fillna(0, inplace=True)

In [289]:
# Calculate Delta (Expected Inflow - Expected Outflow - Missed Demand)
for vehicle in ordered_available_vehicle:
    result["Delta_{}".format(vehicle)] = result["Inflow_{}".format(vehicle)] - result["OutFlow_Fulfilled_{}".format(vehicle)] - result["OutFlow_Missed_{}".format(vehicle)]

In [298]:
estimate_delta_demand(trips)

/Users/demircanm/opt/anaconda3/envs/SharedMobility/lib/python3.7/site-packages/ipykernel_launcher.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  # This is added back by InteractiveShellApp.init_path()
/Users/demircanm/opt/anaconda3/envs/SharedMobility/lib/python3.7/site-packages/pandas/core/indexing.py:1765: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  isetter(loc, value)
/Users/demircanm/opt/anaconda3/envs/SharedMobility/lib/python3.7/site-packages/pandas/core/indexing.py:1765: SettingWithCopyWarnin

,hex_id,index,Inflow_SharedCar,Inflow_SharedKickScooter,Inflow_SharedScooter,OutFlow_Fulfilled_SharedCar,OutFlow_Missed_SharedCar,OutFlow_Fulfilled_SharedKickScooter,OutFlow_Missed_SharedKickScooter,OutFlow_Fulfilled_SharedScooter,OutFlow_Missed_SharedScooter,Delta_SharedKickScooter,Delta_SharedScooter,Delta_SharedCar
0,871f1d4d2ffffff,112,148.0,72.0,229.0,130.0,69.0,75.0,57.0,240.0,2.0,-60.0,-13.0,-51.0
1,871f18b24ffffff,20,111.0,71.0,180.0,135.0,51.0,65.0,37.0,182.0,19.0,-31.0,-21.0,-75.0
2,871f18b20ffffff,12,24.0,1.0,21.0,12.0,7.0,1.0,5.0,17.0,12.0,-5.0,-8.0,5.0
3,871f1d4d4ffffff,116,89.0,64.0,192.0,98.0,18.0,61.0,33.0,157.0,12.0,-30.0,23.0,-27.0
4,871f1d48bffffff,64,201.0,298.0,381.0,216.0,29.0,313.0,19.0,384.0,19.0,-34.0,-22.0,-44.0
5,871f1d488ffffff,58,158.0,256.0,303.0,183.0,18.0,270.0,37.0,291.0,17.0,-51.0,-5.0,-43.0
6,871f1d489ffffff,60,200.0,441.0,476.0,218.0,39.0,444.0,11.0,505.0,3.0,-14.0,-32.0,-57.0
7,871f1d48affffff,62,135.0,173.0,281.0,174.0,22.0,188.0,23.0,267.0,17.0,-38.0,-3.0,-61.0
8,871f1d4f3ffffff,136,32.0,4.0,31.0,28.0,13.0,1.0,12.0,16.0,9.0,-9.0,6.0,-9.0
9,871f1d498ffffff,74,71.0,18.0,80.0,60.0,7.0,22.0,29.0,67.0,26.0,-33.0,-13.0,4.0


In [297]:
def estimate_delta_demand(transactions, ordered_available_vehicle = ["SharedKickScooter", "SharedScooter", "SharedCar"]):
    """
        Compute the expected demand in a hexagon based on collected rentral transactions from the simulation. 
        
        Transactions: Transaction Records from the Payment Interface
        Ordered_Available_Vehicle: List with Available Vehicles and Natural Ordering of Preference in case of neutrality
            i.e., the following hodls 
                ordered_available_vehicle[i] = 1 and ordered_available_vehicle[k] = 1
                            
                                        k > i 
                                        
                => 
                
                Demand is counted for: ordered_available_vehicle[i] as we assume preference order i << k 
        
        We calculate for each vehicle type -- based on the previous assumtion -- the delta demand:
        
                E[Delta Demand| Hexagon ID] = E[Inflow | Hexagon ID] - E[Outflow s| Hexagon ID] - E[Missed Trips | Hexagon ID]
    """
    ordered_available_vehicle = ["SharedKickScooter", "SharedScooter", "SharedCar"]
    # Rearrange Columns according to Vehicle Preferences
    cols = transactions.columns.tolist()
    cols = cols[:-(len(ordered_available_vehicle))] + ordered_available_vehicle
    cols

    # Rearrange Columns
    transactions = transactions[cols]

    transactions["cheapest_vehicle_mode"] = ''

    for i in range(len(ordered_available_vehicle)):
        query = '{} == {}'.format(ordered_available_vehicle[i], 1)
        for k in range(i):
            query += ' & {} == {}'.format(ordered_available_vehicle[k], 0)

        transactions.loc[transactions.query(query).index, "cheapest_vehicle_mode"] = ordered_available_vehicle[i]    
        

    # (All) Hexagons
    hexagons = pd.unique(transactions[["origin", "destination"]].values.ravel('K'))
    hexagons = pd.DataFrame(hexagons, columns=["hex_id"])
    
    # Calculate Expected Outflow 
    outflow = transactions.groupby(["origin", "state", "cheapest_vehicle_mode"]).agg({"destination":"count"})

    # Unstack Vehicle Type Level 
    outflow = outflow.unstack()
    vehicle_columns = list(outflow["destination"].columns.values)
    # Reset Multi-Index to Flat Index
    outflow = outflow.reset_index()
    outflow.columns = ["origin", "state"] + vehicle_columns

    # to ensure that "fulfilled" comes before "missed" during unstacking
    outflow = outflow.sort_values(["origin", "state"])

    # Unstack State Column
    outflow = outflow.set_index(["origin", "state"])
    outflow = outflow.unstack().reset_index()

    # Arrange Column Names
    unfolded_cols = ["hex_id"] 
    for vehicle in vehicle_columns:
        unfolded_cols = unfolded_cols + ["OutFlow_Fulfilled_{}".format(vehicle), "OutFlow_Missed_{}".format(vehicle)]

    # Overwrite Column Names
    outflow.columns = unfolded_cols
    
    # Calculate Expected Inflow
    inflow = transactions.groupby(["destination", "state", "cheapest_vehicle_mode"]).agg({"origin":"count"})

    # Unstack Vehicle Type Level
    inflow = inflow.unstack()
    base_column = ["destination", "state"]
    vehicle_columns = list(inflow["origin"].columns.values)

    # Reset Multi-Index to Flat Index (keep vehicle column order)
    inflow = inflow.reset_index()
    inflow.columns = ["hex_id", "state"] + ["Inflow_{}".format(vehicle) for vehicle in vehicle_columns]

    # to ensure that "fulfilled" comes before "missed" during unstacking
    inflow = inflow.sort_values(["hex_id", "state"])

    # Inflow, we just need fulfilled trips to estimate expected inflow
    inflow = inflow[inflow["state"]=="Fulfilled"]
    inflow = inflow.drop(columns={"state"})
    inflow = inflow.reset_index()
    
    # Put together expected outflow and inflow
    result = hexagons.merge(inflow, on = ["hex_id"], how = "left", suffixes = ("",""))
    result = result.merge(outflow, on = ["hex_id"], how = "left", suffixes = ("_inflow", "_outflow"))
    result.fillna(0, inplace=True)
    
    # Calculate Delta (Expected Inflow - Expected Outflow - Missed Demand)
    for vehicle in ordered_available_vehicle:
        result["Delta_{}".format(vehicle)] = result["Inflow_{}".format(vehicle)] - result["OutFlow_Fulfilled_{}".format(vehicle)] - result["OutFlow_Missed_{}".format(vehicle)]
        
    return result

# Part 2 B: Identify Low SOC Vehicles 

In [2]:
# Pandas Config
pd.set_option('display.max_columns', None)
pd.options.display.max_rows = 200

# ShareNow Data (results from Prepare_Trip_Data)
FLEET_STATE = '../../data/output/fleet_state.csv'

DISTANCE_MATRIX = '../../data/infrastructure/distance_matrix_hex_7.csv'

# TIMEZONE
TIMEZONE = 'Europe/Berlin'

In [3]:
# Read Trips
state = pd.read_csv(FLEET_STATE)

In [4]:
# Distance Matrix between Hexagons
distance_matrix = pd.read_csv(DISTANCE_MATRIX, index_col= 0)

In [5]:
state

,hex_id,Inflow_SharedCar,Inflow_SharedKickScooter,Inflow_SharedScooter,OutFlow_Fulfilled_SharedCar,OutFlow_Missed_SharedCar,OutFlow_Fulfilled_SharedKickScooter,OutFlow_Missed_SharedKickScooter,OutFlow_Fulfilled_SharedScooter,OutFlow_Missed_SharedScooter,Delta_SharedKickScooter,Delta_SharedScooter,Delta_SharedCar,SharedKickScooter,SharedScooter,SharedCar,SharedKickScooter_Low_SoC,SharedScooter_Low_SoC,SharedCar_Low_SoC,Residual_Supply_SharedKickScooter,Residual_Supply_SharedScooter,Residual_Supply_SharedCar
0,871f1d4d2ffffff,28.0,71.0,164.0,15.0,185.0,73.0,59.0,202.0,39.0,-61.0,-77.0,-172.0,18.0,48.0,22.0,7.0,5.0,0.0,-50.0,-34.0,-150.0
1,871f1d4d6ffffff,29.0,163.0,232.0,40.0,139.0,154.0,43.0,234.0,89.0,-34.0,-91.0,-150.0,52.0,43.0,52.0,12.0,9.0,3.0,6.0,-57.0,-101.0
2,871f18b24ffffff,16.0,66.0,113.0,27.0,161.0,61.0,42.0,131.0,73.0,-37.0,-91.0,-172.0,28.0,25.0,16.0,5.0,5.0,0.0,-14.0,-71.0,-156.0
3,871f18b20ffffff,5.0,1.0,14.0,3.0,16.0,1.0,5.0,10.0,19.0,-5.0,-15.0,-14.0,1.0,8.0,4.0,0.0,3.0,0.0,-4.0,-10.0,-10.0
4,871f1d4d4ffffff,18.0,66.0,128.0,17.0,101.0,62.0,32.0,119.0,50.0,-28.0,-41.0,-100.0,27.0,41.0,24.0,5.0,4.0,1.0,-6.0,-4.0,-77.0
5,871f1d48bffffff,68.0,286.0,294.0,51.0,197.0,305.0,27.0,277.0,130.0,-46.0,-113.0,-180.0,103.0,59.0,73.0,16.0,10.0,4.0,41.0,-64.0,-111.0
6,871f1d488ffffff,57.0,254.0,208.0,41.0,160.0,267.0,39.0,191.0,119.0,-52.0,-102.0,-144.0,94.0,57.0,53.0,16.0,8.0,2.0,26.0,-53.0,-93.0
7,871f1d489ffffff,41.0,434.0,351.0,44.0,215.0,437.0,18.0,337.0,175.0,-21.0,-161.0,-218.0,154.0,91.0,77.0,40.0,13.0,6.0,93.0,-83.0,-147.0
8,871f1d48affffff,30.0,173.0,192.0,43.0,155.0,184.0,27.0,178.0,107.0,-38.0,-93.0,-168.0,77.0,40.0,42.0,9.0,10.0,2.0,30.0,-63.0,-128.0
9,871f1d4f3ffffff,10.0,3.0,21.0,7.0,34.0,1.0,12.0,11.0,14.0,-10.0,-4.0,-31.0,2.0,15.0,7.0,0.0,2.0,0.0,-8.0,9.0,-24.0


In [224]:
state.sort_values(by=[f"{mode}_Low_SoC"], ascending=False)



,hex_id,Inflow_SharedCar,Inflow_SharedKickScooter,Inflow_SharedScooter,OutFlow_Fulfilled_SharedCar,OutFlow_Missed_SharedCar,OutFlow_Fulfilled_SharedKickScooter,OutFlow_Missed_SharedKickScooter,OutFlow_Fulfilled_SharedScooter,OutFlow_Missed_SharedScooter,Delta_SharedKickScooter,Delta_SharedScooter,Delta_SharedCar,SharedKickScooter,SharedScooter,SharedCar,SharedKickScooter_Low_SoC,SharedScooter_Low_SoC,SharedCar_Low_SoC,Residual_Supply_SharedKickScooter,Residual_Supply_SharedScooter,Residual_Supply_SharedCar
7,871f1d489ffffff,41.0,434.0,351.0,44.0,215.0,437.0,18.0,337.0,175.0,-21.0,-161.0,-218.0,154.0,91.0,77.0,40.0,13.0,6.0,93.0,-83.0,-147.0
17,871f1d49dffffff,56.0,293.0,194.0,63.0,219.0,301.0,45.0,184.0,169.0,-53.0,-159.0,-226.0,105.0,42.0,69.0,24.0,8.0,5.0,28.0,-125.0,-162.0
26,871f1d48effffff,30.0,147.0,104.0,32.0,115.0,141.0,34.0,105.0,95.0,-28.0,-96.0,-117.0,57.0,33.0,30.0,17.0,7.0,5.0,12.0,-70.0,-92.0
5,871f1d48bffffff,68.0,286.0,294.0,51.0,197.0,305.0,27.0,277.0,130.0,-46.0,-113.0,-180.0,103.0,59.0,73.0,16.0,10.0,4.0,41.0,-64.0,-111.0
41,871f1d48cffffff,26.0,35.0,83.0,20.0,105.0,32.0,25.0,97.0,41.0,-22.0,-55.0,-99.0,20.0,25.0,20.0,2.0,1.0,3.0,-4.0,-31.0,-82.0
1,871f1d4d6ffffff,29.0,163.0,232.0,40.0,139.0,154.0,43.0,234.0,89.0,-34.0,-91.0,-150.0,52.0,43.0,52.0,12.0,9.0,3.0,6.0,-57.0,-101.0
36,871f18b22ffffff,14.0,2.0,16.0,8.0,75.0,4.0,19.0,15.0,41.0,-21.0,-40.0,-69.0,0.0,6.0,12.0,0.0,2.0,2.0,-21.0,-36.0,-59.0
43,871f1d48dffffff,24.0,97.0,118.0,27.0,101.0,94.0,20.0,116.0,68.0,-17.0,-66.0,-104.0,57.0,37.0,21.0,5.0,1.0,2.0,35.0,-30.0,-85.0
24,871f1d499ffffff,24.0,85.0,95.0,28.0,149.0,72.0,37.0,97.0,104.0,-24.0,-106.0,-153.0,33.0,28.0,30.0,7.0,4.0,2.0,2.0,-82.0,-125.0
10,871f1d498ffffff,16.0,17.0,52.0,13.0,54.0,19.0,32.0,46.0,48.0,-34.0,-42.0,-51.0,11.0,14.0,20.0,3.0,2.0,2.0,-26.0,-30.0,-33.0


In [6]:
available_modes = ["SharedKickScooter", "SharedScooter", "SharedCar"]

max_vehicle_supply  = [1000, 1000, 1000]



In [7]:
def calculate_resource_capacity_split(fleet_state, available_modes, max_vehicle_supply, number_of_service_units):
    """
        Determine Service Worker Split Between Modes (How many workers should be assigned to charge how many vehicles)
    """
    
    current_low_soc_supply = []

    for mode in available_modes: 
        
        # Service Worker Charging Capacity per Mode  
        max_capacity_per_unit = get_max_capacity_per_unit(mode)
        
        # Normalize the vehicle weight; 
        # otherwise all resources are bound to kick scooters as there are way more low soc kick scooters
        # One Service Worker can "process" 40 units of kick scooters compared to 1 sw for one car.
        current_low_soc_supply.append(fleet_state[f"{mode}_Low_SoC"].sum()/max_capacity_per_unit)

    low_fraction_per_mode = [curr_low_supply/max_supply for curr_low_supply, max_supply in zip(current_low_soc_supply, max_vehicle_supply)]

    # Temporary Denom
    denominator = sum(low_fraction_per_mode)

    # Split capacity based on the following fraction
    capacity_fraction = low_fraction_per_mode/denominator

    capacity = capacity_fraction*number_of_service_units

    capacity = [round(x) for x in capacity]

    # Rounding Fractions => Sum of Residual don't add up to available units
    # We fix this by adding or subtracting the residual to the max. element
    # Happens very rarely
    residual = number_of_service_units - sum(capacity)

    max_element = max(capacity)
    arg_max = capacity.index(max_element)

    if residual > 0:    
        # We add excess supply to mode where most workforce is needed
        capacity[arg_max] += residual
    elif residual < 0: 
        capacity[arg_max] = max(capacity[arg_max] + residual, 0)
    
    return capacity

In [24]:
def get_max_capacity_per_unit(mode):
    # Mode Specific Collection Capacity per Service Worker (i.e., how many vehicles can one sw collect)
    if mode == "SharedKickScooter":
        max_capacity_per_unit = 40
    elif mode == "SharedScooter":
        max_capacity_per_unit = 8
    elif mode == "SharedCar":    
        max_capacity_per_unit = 1
    else:
        raise RunTimeError("Not Implemented")
    return max_capacity_per_unit

In [11]:
available_modes = ["SharedKickScooter", "SharedScooter", "SharedCar"]

max_vehicle_supply  = [1000, 1000, 1000]

#### Input

number_of_service_units = 100


distance_matrix

state

# Potential Fleet Base Regions for charging 
# (can handle multiple bases e.g.,  base_regions = ['871f1d488ffffff','871f18b21ffffff'])
# '871f1d488ffffff' == Berlin Mitte (Unter den Linden, Deutscher Bundestag)
base_regions = ['871f1d488ffffff']

# Set Base Capacity to 1000 vehicles
base_capacity = 1000
free_base_capacity = dict(zip(base_regions, [base_capacity]*len(base_regions)))


#### Rule Based Charging Process

# Calculate how many workers should be assigned to charge specific vehicle types
# e.g., [1, 3, 20] means 1 => 1 Service Worker for Kick Scooter, 
capacity_per_mode = calculate_resource_capacity_split(state, available_modes, max_vehicle_supply, number_of_service_units)

print(capacity_per_mode)
# We calculate for each vehicle mode dedicated charging plans based on the available resources
all_charging_plans = {}

for i, mode in enumerate(available_modes):
    # Sort State by Regions with highest number of SoC (per vehicle)
    state = state.sort_values(by=[f"{mode}_Low_SoC"], ascending=False)
        
    # Get Max Charging Capacity per Service Worker Unit
    # e.g., 40 Kick Scooters by one SW, 1 Car by one SW
    max_capacity_per_unit = get_max_capacity_per_unit(mode)
        
    # Mode Specific Service Unit Resources (i.e., how many service workers have we assigned to collect the current mode)
    number_of_service_units_per_mode = capacity_per_mode[i]
    
    # Keep track of vehicles which are already scheduled for pick up and charging per region
    already_scheduled_low_vehicles = {}

    # Resulting charging plan having charging tasks per service worker
    # Each element represents one pick up and charging task
    # each element can consist of multiple tasks since vehicles from nearby regions can also be collected if 
    # space if available.
    # Example [(origin, destination, number of vehicles to pick up, destination to base)]
    # Multitask Example 
    # [
    #  ('871f1d49dffffff', '871f1d488ffffff', 24.0, 4173.489610482256),
    #  ('871f1d48effffff', '871f1d488ffffff', 16.0, 2312.641785385671) <- This is the neighbour of 871f1d49dffffff, distance to neighbour
    # ]
    # Note that the first element has the distance to the base and all other elements are relative to the first (main) region. 
    charging_plan = []

    for index, row in state.iterrows():
        next_hex_id = row["hex_id"]
        
        # Break loop if service worker resources are depleted
        if len(charging_plan) == number_of_service_units_per_mode:
            # No more service worker capacity, stop planning
            break

        # Keep Track of how many vehicles are already scheduled for charging 
        currently_schedules_vehicles_per_unit = 0

        # Get nearest fleet base region (.loc -> destination, [[]] -> Origin)
        sorted_base_list = distance_matrix.loc[base_regions][[next_hex_id]].sort_values(by = next_hex_id, ascending=True)

        # Select Nearest Base
        for index_base, row_base in sorted_base_list.iterrows():
            selected_base = index_base
            selected_base_distance = row_base[next_hex_id]

        # How many vehicles are already schedules for pick up?
        if next_hex_id not in already_scheduled_low_vehicles:
            already_scheduled_low_vehicles[next_hex_id] = 0


        # How many vehicles can we move to the base and charge?
        pick_up_vehicles = min((row[f"{mode}_Low_SoC"] - already_scheduled_low_vehicles[next_hex_id]), max_capacity_per_unit)

        # currently_schedules_vehicles_per_unit
        currently_schedules_vehicles_per_unit += pick_up_vehicles

        # Update Already Picked Up Vehicle Dict
        already_scheduled_low_vehicles[next_hex_id] += pick_up_vehicles

        # Collection of Charging Tasks per Service Worker
        chargings_tasks = []

        # Charging Task from current region to fleet base 
        charging_task = (next_hex_id, selected_base, pick_up_vehicles, selected_base_distance)

        # Add main chargin task to charging tasks
        if pick_up_vehicles > 0: 
            chargings_tasks.append(charging_task)

        # If we still have capacity, 
        if pick_up_vehicles < max_capacity_per_unit and pick_up_vehicles > 0:
            neighbours = h3.hex_range(next_hex_id, 1)
            neighbours.remove(next_hex_id) # Consider only neighbours (not current region)
            
            # Consider only Fleet State of neighbour regions
            neighbour_states = state[state["hex_id"].isin(neighbours)]
            
            for index_adjacent, row_adjacent in neighbour_states.iterrows():
                adjacent_hex_id = row_adjacent["hex_id"]

                # How many vehicles from the adjacent regions are already schedules for pick up?
                if adjacent_hex_id not in already_scheduled_low_vehicles:
                    already_scheduled_low_vehicles[adjacent_hex_id] = 0

                # How many vehicles can we move from the adjacent hex to the base and charge them?
                pick_up_vehicles_from_adjacent_region = min((row_adjacent[f"{mode}_Low_SoC"] - already_scheduled_low_vehicles[adjacent_hex_id]), max_capacity_per_unit - currently_schedules_vehicles_per_unit)
                                
                # Can we pick up any vehicles from that neighbour? If yes, append charging task
                if pick_up_vehicles_from_adjacent_region > 0:
                    # currently_schedules_vehicles_per_unit
                    currently_schedules_vehicles_per_unit += pick_up_vehicles_from_adjacent_region

                    # Update Already Picked Up Vehicle Dict
                    already_scheduled_low_vehicles[adjacent_hex_id] += pick_up_vehicles_from_adjacent_region

                    # Distance to Neighbour
                    distance_to_neighbour = distance_matrix.loc[adjacent_hex_id][next_hex_id]

                    # add additional charging task
                    charging_task = (adjacent_hex_id, selected_base, pick_up_vehicles_from_adjacent_region, distance_to_neighbour)

                    # Add to Charging Tasks list (i.e., this is a multitask object; one service workers needs to visit multiple regions)
                    chargings_tasks.append(charging_task)


        # Store Charging Task
        if len(chargings_tasks) > 0:
            charging_plan.append(chargings_tasks)
    
    # Save Charging Plan of Vehicle Mode
    all_charging_plans[mode] = charging_plan
    
    # Count planned resources (e.g., service workers needed to execute planned charging jobs)
    used_resources = 0
    for key in all_charging_plans:
        used_resources += len(all_charging_plans[key])
    
    
print(all_charging_plans)
print(used_resources)

# TODO: Create OperatorTask, ChargingTask, RelocationTask and Service Worker to Process these Tasks


[7, 23, 70]
{'SharedKickScooter': [[('871f1d489ffffff', '871f1d488ffffff', 40.0, 2405.9623599382885)], [('871f1d49dffffff', '871f1d488ffffff', 24.0, 4173.489610482256), ('871f1d48effffff', '871f1d488ffffff', 16.0, 2312.641785385671)], [('871f1d48effffff', '871f1d488ffffff', 1.0, 2405.7473700730616), ('871f1d488ffffff', '871f1d488ffffff', 16.0, 2405.7473700730616), ('871f1d48affffff', '871f1d488ffffff', 9.0, 2203.326411276209), ('871f1d483ffffff', '871f1d488ffffff', 4.0, 2405.5317230538103), ('871f1d481ffffff', '871f1d488ffffff', 2.0, 2202.6294978838255), ('871f1d48cffffff', '871f1d488ffffff', 2.0, 2312.117549732313)], [('871f1d48bffffff', '871f1d488ffffff', 16.0, 2203.6149432051543), ('871f1d4d6ffffff', '871f1d488ffffff', 12.0, 2406.3917177752005), ('871f18b25ffffff', '871f1d488ffffff', 7.0, 2204.311494804113), ('871f18b24ffffff', '871f1d488ffffff', 5.0, 2313.065221687349)], [('871f1d4d2ffffff', '871f1d488ffffff', 7.0, 5985.216472857151), ('871f1d4d0ffffff', '871f1d488ffffff', 7.0, 231

# Rule Based Relocation Strategy with Filter Operator

In [4]:
# Pandas Config
pd.set_option('display.max_columns', None)
pd.options.display.max_rows = 200

# ShareNow Data (results from Prepare_Trip_Data)
FLEET_STATE = '../../data/output/fleet_state.csv'

DISTANCE_MATRIX = '../../data/infrastructure/distance_matrix_hex_7.csv'

# TIMEZONE
TIMEZONE = 'Europe/Berlin'

In [5]:
# Read Trips
state = pd.read_csv(FLEET_STATE)

In [6]:
# Distance Matrix between Hexagons
distance_matrix = pd.read_csv(DISTANCE_MATRIX, index_col= 0)

In [7]:
state

,hex_id,Inflow_SharedCar,Inflow_SharedKickScooter,Inflow_SharedScooter,OutFlow_Fulfilled_SharedCar,OutFlow_Missed_SharedCar,OutFlow_Fulfilled_SharedKickScooter,OutFlow_Missed_SharedKickScooter,OutFlow_Fulfilled_SharedScooter,OutFlow_Missed_SharedScooter,Delta_SharedKickScooter,Delta_SharedScooter,Delta_SharedCar,SharedKickScooter,SharedScooter,SharedCar,SharedKickScooter_Low_SoC,SharedScooter_Low_SoC,SharedCar_Low_SoC,Residual_Supply_SharedKickScooter,Residual_Supply_SharedScooter,Residual_Supply_SharedCar
0,871f18b24ffffff,126.0,10.0,136.0,143.0,45.0,11.0,96.0,132.0,64.0,-97.0,-60.0,-62.0,2,5,13,0,0,0,-95.0,-55.0,-49.0
1,871f1d488ffffff,150.0,91.0,224.0,174.0,31.0,88.0,191.0,234.0,101.0,-188.0,-111.0,-55.0,13,3,12,0,0,0,-175.0,-108.0,-43.0
2,871f1d48bffffff,199.0,111.0,283.0,209.0,33.0,118.0,205.0,304.0,116.0,-212.0,-137.0,-43.0,8,4,18,0,0,0,-204.0,-133.0,-25.0
3,871f1d48affffff,147.0,47.0,211.0,185.0,14.0,50.0,146.0,211.0,86.0,-149.0,-86.0,-52.0,6,5,11,0,0,0,-143.0,-81.0,-41.0
4,871f1d498ffffff,66.0,3.0,63.0,56.0,11.0,3.0,56.0,58.0,27.0,-56.0,-22.0,-1.0,2,1,33,0,0,0,-54.0,-21.0,32.0
5,871f1d489ffffff,208.0,156.0,337.0,236.0,21.0,170.0,308.0,346.0,142.0,-322.0,-151.0,-49.0,11,4,21,0,0,0,-311.0,-147.0,-28.0
6,871f18b20ffffff,25.0,1.0,17.0,13.0,9.0,0.0,10.0,12.0,10.0,-9.0,-5.0,3.0,0,2,19,0,0,0,-9.0,-3.0,22.0
7,871f18b2cffffff,40.0,1.0,32.0,41.0,21.0,1.0,13.0,21.0,26.0,-13.0,-15.0,-22.0,0,1,15,0,0,0,-13.0,-14.0,-7.0
8,871f1d4d4ffffff,81.0,20.0,135.0,97.0,26.0,20.0,65.0,117.0,54.0,-65.0,-36.0,-42.0,7,6,15,0,0,1,-58.0,-30.0,-28.0
9,871f1d4d2ffffff,137.0,23.0,159.0,157.0,47.0,15.0,113.0,144.0,96.0,-105.0,-81.0,-67.0,4,1,35,0,0,0,-101.0,-80.0,-32.0


In [26]:

mode = "SharedKickScooter"

state[f"Filtered_Excess_Supply_{mode}"] = 0 
state[f"Filtered_Lack_Supply_{mode}"] = 0


for index, row in state.iterrows():
    next_hex_id   = row["hex_id"]
    next_residual = row[f"Residual_Supply_{mode}"]
    
    max_capacity_per_unit = get_max_capacity_per_unit(mode)
    
    # Calculate Region Neighbours
    if max_capacity_per_unit > 1: 
        neighbours = h3.hex_range(next_hex_id, 1)
    else:
        neighbours = h3.hex_range(next_hex_id, 0)


        
    # Apply Mathematical Filter Operations over Hexagons (Hexagon + Direct Neighbours) 
    excess_supply = state[(state[f"Residual_Supply_{mode}"] > 0) & (state["hex_id"].isin(neighbours))][f"Residual_Supply_{mode}"].sum()
    lack_supply = state[(state[f"Residual_Supply_{mode}"] < 0) & (state["hex_id"].isin(neighbours))][f"Residual_Supply_{mode}"].sum()
    
    state.loc[state["hex_id"]== next_hex_id, f"Filtered_Excess_Supply_{mode}"] = excess_supply
    state.loc[state["hex_id"]== next_hex_id, f"Filtered_Lack_Supply_{mode}"] = lack_supply


    # print(lack_supply)
    #for index_adjacent, row_adjacent in neighbour_states.iterrows():
    #   adjacent_hex_id = row_adjacent["hex_id"]



In [27]:
state

,hex_id,Inflow_SharedCar,Inflow_SharedKickScooter,Inflow_SharedScooter,OutFlow_Fulfilled_SharedCar,OutFlow_Missed_SharedCar,OutFlow_Fulfilled_SharedKickScooter,OutFlow_Missed_SharedKickScooter,OutFlow_Fulfilled_SharedScooter,OutFlow_Missed_SharedScooter,Delta_SharedKickScooter,Delta_SharedScooter,Delta_SharedCar,SharedKickScooter,SharedScooter,SharedCar,SharedKickScooter_Low_SoC,SharedScooter_Low_SoC,SharedCar_Low_SoC,Residual_Supply_SharedKickScooter,Residual_Supply_SharedScooter,Residual_Supply_SharedCar,Filtered_Excess_Supply_SharedKickScooter,Filtered_Lack_Supply_SharedKickScooter
0,871f18b24ffffff,126.0,10.0,136.0,143.0,45.0,11.0,96.0,132.0,64.0,-97.0,-60.0,-62.0,2,5,13,0,0,0,-95.0,-55.0,-49.0,0.0,-686.0
1,871f1d488ffffff,150.0,91.0,224.0,174.0,31.0,88.0,191.0,234.0,101.0,-188.0,-111.0,-55.0,13,3,12,0,0,0,-175.0,-108.0,-43.0,0.0,-1121.0
2,871f1d48bffffff,199.0,111.0,283.0,209.0,33.0,118.0,205.0,304.0,116.0,-212.0,-137.0,-43.0,8,4,18,0,0,0,-204.0,-133.0,-25.0,0.0,-1203.0
3,871f1d48affffff,147.0,47.0,211.0,185.0,14.0,50.0,146.0,211.0,86.0,-149.0,-86.0,-52.0,6,5,11,0,0,0,-143.0,-81.0,-41.0,0.0,-1159.0
4,871f1d498ffffff,66.0,3.0,63.0,56.0,11.0,3.0,56.0,58.0,27.0,-56.0,-22.0,-1.0,2,1,33,0,0,0,-54.0,-21.0,32.0,0.0,-576.0
5,871f1d489ffffff,208.0,156.0,337.0,236.0,21.0,170.0,308.0,346.0,142.0,-322.0,-151.0,-49.0,11,4,21,0,0,0,-311.0,-147.0,-28.0,0.0,-1136.0
6,871f18b20ffffff,25.0,1.0,17.0,13.0,9.0,0.0,10.0,12.0,10.0,-9.0,-5.0,3.0,0,2,19,0,0,0,-9.0,-3.0,22.0,0.0,-367.0
7,871f18b2cffffff,40.0,1.0,32.0,41.0,21.0,1.0,13.0,21.0,26.0,-13.0,-15.0,-22.0,0,1,15,0,0,0,-13.0,-14.0,-7.0,0.0,-235.0
8,871f1d4d4ffffff,81.0,20.0,135.0,97.0,26.0,20.0,65.0,117.0,54.0,-65.0,-36.0,-42.0,7,6,15,0,0,1,-58.0,-30.0,-28.0,0.0,-850.0
9,871f1d4d2ffffff,137.0,23.0,159.0,157.0,47.0,15.0,113.0,144.0,96.0,-105.0,-81.0,-67.0,4,1,35,0,0,0,-101.0,-80.0,-32.0,0.0,-655.0
